In [1]:
!pip install /kaggle/input/datasets/amirrezaaleyasin/rdkit-2025-9-5/rdkit-2025.9.5-cp312-cp312-manylinux_2_28_x86_64.whl

Processing /kaggle/input/datasets/amirrezaaleyasin/rdkit-2025-9-5/rdkit-2025.9.5-cp312-cp312-manylinux_2_28_x86_64.whl


In [2]:
import warnings
warnings.filterwarnings('ignore')
import os
os.environ['PYTHONWARNINGS'] = 'ignore'
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')

import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import AllChem
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import KFold
from scipy.sparse import csr_matrix, hstack
import lightgbm as lgb

# ── Пути к данным (Kaggle) ──
BASE = '/kaggle/input/competitions/neurips-open-polymer-prediction-2025'

train = pd.read_csv(f'{BASE}/train.csv')
test  = pd.read_csv(f'{BASE}/test.csv')
sub   = pd.read_csv(f'{BASE}/sample_submission.csv')

ds1 = pd.read_csv(f'{BASE}/train_supplement/dataset1.csv')  # Tc
ds3 = pd.read_csv(f'{BASE}/train_supplement/dataset3.csv')  # Tg
ds4 = pd.read_csv(f'{BASE}/train_supplement/dataset4.csv')  # FFV

print('Train shape:', train.shape)
print('Test shape: ', test.shape)
print('\nПропуски (%):')
print((train.isnull().mean() * 100).round(1))

Train shape: (7973, 7)
Test shape:  (3, 2)

Пропуски (%):
id          0.0
SMILES      0.0
Tg         93.6
FFV        11.8
Tc         90.8
Density    92.3
Rg         92.3
dtype: float64


In [3]:
# ── Канонизация SMILES ──
def canonicalize(smi):
    try:
        return Chem.MolToSmiles(Chem.MolFromSmiles(smi))
    except:
        return None

train['canon'] = train['SMILES'].apply(canonicalize)
test['canon']  = test['SMILES'].apply(canonicalize)
ds1['canon']   = ds1['SMILES'].apply(canonicalize)
ds3['canon']   = ds3['SMILES'].apply(canonicalize)
ds4['canon']   = ds4['SMILES'].apply(canonicalize)

ds1 = ds1.rename(columns={'TC_mean': 'Tc'})

train_canons = set(train['canon'].dropna())
ds1_new = ds1[~ds1['canon'].isin(train_canons)][['canon', 'Tc']]
ds3_new = ds3[~ds3['canon'].isin(train_canons)][['canon', 'Tg']]
ds4_new = ds4[~ds4['canon'].isin(train_canons)][['canon', 'FFV']]

print(f'Новых примеров из supplement: Tc={len(ds1_new)}, Tg={len(ds3_new)}, FFV={len(ds4_new)}')

Новых примеров из supplement: Tc=130, Tg=46, FFV=862


In [4]:
# ── Morgan Fingerprints ──
def smiles_to_morgan(smiles_list, radius=2, n_bits=2048):
    fps = []
    for smi in smiles_list:
        mol = Chem.MolFromSmiles(smi) if smi else None
        if mol:
            fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=n_bits)
            fps.append(np.array(fp))
        else:
            fps.append(np.zeros(n_bits))
    return np.array(fps)

X_train = smiles_to_morgan(train['canon'])
X_test  = smiles_to_morgan(test['canon'])
print('X_train shape:', X_train.shape)
print('X_test shape: ', X_test.shape)

X_train shape: (7973, 2048)
X_test shape:  (3, 2048)


In [5]:
# ── TF-IDF (char n-grams) ──
train_smiles = train['canon'].fillna('').tolist()
test_smiles  = test['canon'].fillna('').tolist()

vectorizer = TfidfVectorizer(
    analyzer='char_wb',
    ngram_range=(2, 4),
    max_features=4096,
    sublinear_tf=True
)
vectorizer.fit(train_smiles + test_smiles)

X_train_tfidf = vectorizer.transform(train_smiles)
X_test_tfidf  = vectorizer.transform(test_smiles)

# ── Combined: Morgan + TF-IDF ──
X_train_combined = hstack([csr_matrix(X_train), X_train_tfidf])
X_test_combined  = hstack([csr_matrix(X_test),  X_test_tfidf])

print('TF-IDF shape:  ', X_train_tfidf.shape)
print('Combined shape:', X_train_combined.shape)

TF-IDF shape:   (7973, 4096)
Combined shape: (7973, 6144)


In [6]:
# ── Обучение LightGBM (3 варианта фич) ──
targets = ['Tg', 'FFV', 'Tc', 'Density', 'Rg']
PROPERTY_WEIGHTS = {'Tg': 1.0, 'FFV': 10.0, 'Tc': 1.0, 'Density': 1.0, 'Rg': 1.0}

def get_X_y_for_target(target, X_base):
    mask = train[target].notna()
    X = X_base[mask.values]
    y = train.loc[mask, target].values
    supp_map = {'Tc': (ds1_new, 'Tc'), 'Tg': (ds3_new, 'Tg'), 'FFV': (ds4_new, 'FFV')}
    if target in supp_map and X_base.shape[1] == 2048:  # supplement только для Morgan
        supp_df, col = supp_map[target]
        supp = supp_df.dropna(subset=[col])
        X = np.vstack([X, smiles_to_morgan(supp['canon'].tolist())])
        y = np.concatenate([y, supp[col].values])
    return X, y

def train_lgbm(X_features, X_test_features, feature_name):
    results = {}
    all_models = {}
    for target in targets:
        X, y = get_X_y_for_target(target, X_features)
        print(f'\n[{feature_name}] {target}: {len(y)} samples')
        kf = KFold(n_splits=5, shuffle=True, random_state=42)
        oof = np.zeros(len(y))
        fold_models = []
        for tr_idx, val_idx in kf.split(X):
            model = lgb.LGBMRegressor(
                n_estimators=1000, learning_rate=0.05, num_leaves=63,
                subsample=0.8, colsample_bytree=0.8, random_state=42,
                n_jobs=-1, verbose=-1
            )
            model.fit(
                X[tr_idx], y[tr_idx],
                eval_set=[(X[val_idx], y[val_idx])],
                callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)]
            )
            oof[val_idx] = model.predict(X[val_idx])
            fold_models.append(model)
        mae = np.abs(y - oof).mean()
        results[target] = mae
        all_models[target] = fold_models
        print(f'  OOF MAE: {mae:.4f}')
    wmae = sum(results[t] * PROPERTY_WEIGHTS[t] for t in targets) / sum(PROPERTY_WEIGHTS.values())
    results['wMAE'] = wmae
    return results, all_models

print('=' * 50)
print('ПРОГОН 1: Morgan FP')
print('=' * 50)
res_morgan, models_morgan = train_lgbm(X_train, X_test, 'Morgan')

print('\n' + '=' * 50)
print('ПРОГОН 2: TF-IDF')
print('=' * 50)
res_tfidf, models_tfidf = train_lgbm(X_train_tfidf, X_test_tfidf, 'TF-IDF')

print('\n' + '=' * 50)
print('ПРОГОН 3: Morgan + TF-IDF')
print('=' * 50)
res_combined, models_combined = train_lgbm(X_train_combined, X_test_combined, 'Combined')

print('\n' + '=' * 60)
print(f"{'Таргет':<12} {'Morgan':>10} {'TF-IDF':>10} {'Combined':>10}")
print('-' * 60)
for t in targets + ['wMAE']:
    print(f'{t:<12} {res_morgan[t]:>10.4f} {res_tfidf[t]:>10.4f} {res_combined[t]:>10.4f}')
print('=' * 60)

ПРОГОН 1: Morgan FP

[Morgan] Tg: 557 samples
  OOF MAE: 54.1883

[Morgan] FFV: 7892 samples
  OOF MAE: 0.0071

[Morgan] Tc: 867 samples
  OOF MAE: 0.0370

[Morgan] Density: 613 samples
  OOF MAE: 0.0505

[Morgan] Rg: 614 samples
  OOF MAE: 1.8028

ПРОГОН 2: TF-IDF

[TF-IDF] Tg: 511 samples
  OOF MAE: 55.6388

[TF-IDF] FFV: 7030 samples
  OOF MAE: 0.0064

[TF-IDF] Tc: 737 samples
  OOF MAE: 0.0292

[TF-IDF] Density: 613 samples
  OOF MAE: 0.0474

[TF-IDF] Rg: 614 samples
  OOF MAE: 1.6368

ПРОГОН 3: Morgan + TF-IDF

[Combined] Tg: 511 samples
  OOF MAE: 51.3570

[Combined] FFV: 7030 samples
  OOF MAE: 0.0060

[Combined] Tc: 737 samples
  OOF MAE: 0.0285

[Combined] Density: 613 samples
  OOF MAE: 0.0432

[Combined] Rg: 614 samples
  OOF MAE: 1.5793

Таргет           Morgan     TF-IDF   Combined
------------------------------------------------------------
Tg              54.1883    55.6388    51.3570
FFV              0.0071     0.0064     0.0060
Tc               0.0370     0.0292     0.

In [7]:
# ── Сабмит (Combined — лучший) ──
for target in targets:
    sub[target] = np.mean([m.predict(X_test_combined) for m in models_combined[target]], axis=0)

sub.to_csv('submission.csv', index=False)
print('Сабмит сохранён: submission.csv')
print(sub)

Сабмит сохранён: submission.csv
           id          Tg       FFV        Tc   Density         Rg
0  1109053969  208.277265  0.376101  0.207212  1.146820  22.364309
1  1422188626  139.517735  0.377910  0.236761  1.099218  21.444576
2  2032016830   83.766551  0.352022  0.305322  1.116440  21.683565
